# 🥈 04 — Dhara: Silver Layer (Cleaned & Normalized)

Normalizes heterogeneous train-timings CSVs into a clean `silver.train_timings` schema, and builds a `silver.station_master` lookup used for joins with air quality.

The raw CSVs have varied column names (`Name of the Train`, `train_name`, `Train No.`, etc.). We handle this via dynamic column detection.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA silver;

## Step 1 — Normalize train timings

In [0]:
# ================================
# STEP 1: Load tables
# ================================
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("USE CATALOG setu_rail")

sr = spark.table("setu_rail.bronze.sr_timings")
trains = spark.table("setu_rail.bronze.trains")
stations = spark.table("setu_rail.bronze.stations")
aq = spark.table("setu_rail.bronze.air_quality")

print("Loaded tables")
print("SR:", sr.columns)
print("TRAINS:", trains.columns)
print("STATIONS:", stations.columns)
print("AQ:", aq.columns)

## Step 2 — Build station_master (top Indian stations with city mapping)

In [0]:
# ================================
# STEP 2: Basic cleaning
# ================================
sr = sr.dropna(how="all")

# ensure train_no exists
if "train_no" in sr.columns:
    sr = sr.withColumn("train_no", col("train_no").cast("string"))
else:
    raise Exception("train_no column missing in sr_timings")

In [0]:
# ================================
# STEP 3: Feature engineering
# ================================

# find a safe ordering column
order_col = None
for c in sr.columns:
    if "seq" in c.lower() or "order" in c.lower():
        order_col = c
        break

if not order_col:
    order_col = sr.columns[0]

w = Window.partitionBy("train_no").orderBy(order_col)

sr = sr.withColumn("stop_seq", row_number().over(w))

# hour_of_day SAFE
if "departure_time" in sr.columns:
    sr = sr.withColumn("hour_of_day", hour(to_timestamp("departure_time")))
else:
    sr = sr.withColumn("hour_of_day", lit(12))

# peak hour
sr = sr.withColumn(
    "is_peak_hour",
    when(
        (col("hour_of_day").between(7,10)) | 
        (col("hour_of_day").between(17,20)),
        1
    ).otherwise(0)
)

## Step 3 — Enrich train timings with station + AQ features

In [0]:
# ================================
# STEP 4: Safe joins
# ================================

# trains join
if "train_no" in trains.columns:
    sr = sr.join(trains, on="train_no", how="left")

# station join (auto-detect)
station_col = None
for c in sr.columns:
    if "station" in c.lower():
        station_col = c
        break

if station_col and station_col in stations.columns:
    sr = sr.join(stations, on=station_col, how="left")

# AQI join
if "state" in sr.columns and "state" in aq.columns:
    sr = sr.join(aq, on="state", how="left")

In [0]:
# ================================
# STEP 5: AQI feature (SAFE)
# ================================

aq_col = None
for c in sr.columns:
    if "pm" in c.lower():
        aq_col = c
        break

if aq_col:
    sr = sr.withColumn(aq_col, col(aq_col).cast("double"))

    sr = sr.withColumn(
        "fog_factor",
        when(col(aq_col) > 150, 1.5)
        .when(col(aq_col) > 80, 1.2)
        .otherwise(1.0)
    )
else:
    sr = sr.withColumn("fog_factor", lit(1.0))

In [0]:
# ================================
# STEP 6: Final cleanup
# ================================
sr = sr.dropDuplicates()
sr = sr.withColumn("_processed_ts", current_timestamp())

In [0]:
# FIX duplicate columns using selectExpr (WORKS on serverless)

cols = sr.columns
seen = {}

exprs = []

for c in cols:
    if c in seen:
        seen[c] += 1
        new_name = f"{c}_{seen[c]}"
    else:
        seen[c] = 0
        new_name = c

    exprs.append(f"`{c}` as `{new_name}`")

sr = sr.selectExpr(*exprs)

In [0]:
# ================================
# STEP 7: WRITE (FINAL)
# ================================
(sr.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.silver.sr_enriched")
)

print("✅ DONE: setu_rail.silver.sr_enriched")

✅ **Next:** `05_build_gold_features.ipynb`